In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

#  Imports and Configuration

In [ ]:


# Integrated SR + MLC Pipeline

import os, re, json, random, csv, time, math
from pathlib import Path
from collections import defaultdict, Counter
from functools import lru_cache
import numpy as np
from PIL import Image, ImageFilter
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import resnet50, ResNet50_Weights
from tqdm.auto import tqdm
import tifffile
from math import exp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Try torch_geometric
USE_PYG = True
try:
    import torch_geometric
    from torch_geometric.nn import GATConv
    from torch_geometric.data import Data
    from torch_geometric.utils import dense_to_sparse
except Exception:
    print("torch_geometric not available.")
    USE_PYG = False

# Configuration
SR_LR_SIZE = (40, 40)
SR_HR_TARGET_SIZE = (160, 160)
SR_UPSCALE_FACTOR = 4
FEATURE_CHANNELS = 512
LATENT_DIM = 256
FEATURE_MAP_SIZE = 10
MLC_IMG_SIZE = 224
MLC_BATCH_SIZE = 8
MLC_LR = 1e-4
MLC_WEIGHT_DECAY = 1e-5
MLC_DROPOUT = 0.4
SR_PRETRAIN_EPOCHS = 1
MLC_TRAIN_EPOCHS = 2
JOINT_EPOCHS = 3
JOINT_LR = 1e-5
BASE_BETA = 3e-4
LAMBDA_PERCEP = 0.1
LAMBDA_EDGE = 0.05
LAMBDA_SSIM = 0.05
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Paths which we've used
# SENTINEL_ROOT = Path('/kaggle/input/pragnasdataset/FLAIR2 DATASET WITH LABELS/Sentinelpatches')
# AERIAL_ROOT = Path('/kaggle/input/pragnasdataset/FLAIR2 DATASET WITH LABELS/Aerialpatches')
# LABEL_PAIRS_FILE = Path('/kaggle/working/image_pairs_with_labels.json')
# TEST_SENTINEL_ROOT = Path('/kaggle/input/newtestdataset/Test with 14 labels/Sentinelpatches/Z18_UN')
# TEST_AERIAL_ROOT = Path('/kaggle/input/newtestdataset/Test with 14 labels/Aerialpatches')
# TEST_JSON_FILE = Path('/kaggle/working/test_image_pairs_with_labels.json')
# OUT_DIR = Path('/kaggle/working/joint_sr_mlc_results')
# TEST_OUTPUT_DIR = Path('/kaggle/working/test_results_separate_dataset')

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0

print(f"Device: {DEVICE}, PyG available: {USE_PYG}")

# Metrics Functions

In [ ]:
def calculate_psnr(img1, img2):
    if img1.device != img2.device:
        img2 = img2.to(img1.device)
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(1.0 / torch.sqrt(mse))

def calculate_ssim(img1, img2, window_size=11, size_average=True):
    try:
        if img1.device != img2.device:
            img2 = img2.to(img1.device)
        
        _, channel, height, width = img1.size()
        
        def gaussian(window_size, sigma):
            gauss = torch.Tensor([exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
            return gauss/gauss.sum()

        def create_window(window_size, channel):
            _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
            _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
            window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
            return window

        window = create_window(window_size, channel)
        if img1.is_cuda:
            window = window.cuda(img1.get_device())
        window = window.type_as(img1)
        
        mu1 = F.conv2d(img1, window, padding=window_size//2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=window_size//2, groups=channel)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2

        sigma1_sq = F.conv2d(img1*img1, window, padding=window_size//2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, window, padding=window_size//2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, window, padding=window_size//2, groups=channel) - mu1_mu2

        C1 = 0.01**2
        C2 = 0.03**2

        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))

        if size_average:
            return ssim_map.mean()
        else:
            return ssim_map.mean(1).mean(1).mean(1)
    except:
        return torch.tensor(0.0, device=img1.device)

def compute_mlc_metrics(preds, labels):
    eps = 1e-8
    
    if preds.size == 0 or labels.size == 0:
        return {
            'micro_precision': 0.0, 'micro_recall': 0.0, 'micro_f1': 0.0,
            'macro_precision': 0.0, 'macro_recall': 0.0, 'macro_f1': 0.0,
            'exact_match': 0.0, 'hamming_loss': 1.0,
            'per_class_precision': [], 'per_class_recall': [], 'per_class_f1': []
        }
    
    tp = ((preds == 1) & (labels == 1)).sum(axis=0)
    fp = ((preds == 1) & (labels == 0)).sum(axis=0)
    fn = ((preds == 0) & (labels == 1)).sum(axis=0)
    tn = ((preds == 0) & (labels == 0)).sum(axis=0)
    
    precision_per_class = tp / (tp + fp + eps)
    recall_per_class = tp / (tp + fn + eps)
    f1_per_class = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + eps)
    
    micro_p = tp.sum() / (tp.sum() + fp.sum() + eps)
    micro_r = tp.sum() / (tp.sum() + fn.sum() + eps)
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r + eps)
    
    macro_p = precision_per_class.mean()
    macro_r = recall_per_class.mean()
    macro_f1 = f1_per_class.mean()
    
    exact_match = (preds == labels).all(axis=1).mean()
    hamming_loss = (preds != labels).mean()
    
    return {
        'micro_precision': float(micro_p),
        'micro_recall': float(micro_r),
        'micro_f1': float(micro_f1),
        'macro_precision': float(macro_p),
        'macro_recall': float(macro_r),
        'macro_f1': float(macro_f1),
        'exact_match': float(exact_match),
        'hamming_loss': float(hamming_loss),
        'per_class_precision': precision_per_class.tolist(),
        'per_class_recall': recall_per_class.tolist(),
        'per_class_f1': f1_per_class.tolist()
    }

# SR Model Components

In [ ]:
class ImprovedVGGFeatureEncoder(nn.Module):
    def __init__(self, output_channels=FEATURE_CHANNELS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(256, output_channels, kernel_size=3, padding=1), nn.BatchNorm2d(output_channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(output_channels, output_channels, kernel_size=3, padding=1), nn.BatchNorm2d(output_channels), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2)
        )
        
    def forward(self, x):
        return self.features(x)

class FeatureEncoder(nn.Module):
    def __init__(self, feature_channels=FEATURE_CHANNELS, latent_dim=LATENT_DIM, feature_map_size=FEATURE_MAP_SIZE):
        super().__init__()
        input_dim = feature_channels * feature_map_size * feature_map_size
        
        self.net = nn.Sequential(
            nn.Conv2d(feature_channels * 2, feature_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_channels, feature_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten()
        )
        
        self.mu_fc = nn.Linear(input_dim, latent_dim)
        self.logvar_fc = nn.Linear(input_dim, latent_dim)

    def forward(self, feature_R, feature_X):
        x = torch.cat([feature_R, feature_X], dim=1)
        x = self.net(x)
        mu = self.mu_fc(x)
        logvar = self.logvar_fc(x)
        return mu, logvar

class FeatureDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, feature_channels=FEATURE_CHANNELS, feature_map_size=FEATURE_MAP_SIZE):
        super().__init__()
        output_dim = feature_channels * feature_map_size * feature_map_size
        
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, output_dim), nn.LeakyReLU(0.2, inplace=True),
        )
        
        self.conv = nn.Sequential(
            nn.Conv2d(feature_channels, feature_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_channels, feature_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(feature_channels), nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, z):
        batch_size = z.size(0)
        x = self.fc(z)
        x = x.view(batch_size, FEATURE_CHANNELS, FEATURE_MAP_SIZE, FEATURE_MAP_SIZE)
        x = self.conv(x)
        return x

class ImageDecoder(nn.Module):
    def __init__(self, in_channels=FEATURE_CHANNELS, out_channels=3):
        super().__init__()
        
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(in_channels, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True)
        )
        
        self.final = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(16, out_channels, kernel_size=3, padding=1), nn.Tanh()
        )
        
    def forward(self, x):
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)
        x = self.final(x)
        return (x + 1) / 2

class SR_Feature_CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.bicubic_upsample = transforms.Resize(SR_HR_TARGET_SIZE, interpolation=transforms.InterpolationMode.BICUBIC)
        self.vgg_encoder_r = ImprovedVGGFeatureEncoder()
        self.vgg_encoder_x = ImprovedVGGFeatureEncoder()
        self.feature_encoder = FeatureEncoder()
        self.feature_decoder = FeatureDecoder()
        self.image_decoder = ImageDecoder()

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x_lr, x_hr=None):
        x_bicubic = self.bicubic_upsample(x_lr)
        feature_x = self.vgg_encoder_x(x_bicubic)

        if self.training and x_hr is not None:
            feature_r = self.vgg_encoder_r(x_hr)
            mu, logvar = self.feature_encoder(feature_r, feature_x)
            z = self.reparameterize(mu, logvar)
            feature_cr = self.feature_decoder(z)
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
        else:
            z = torch.randn(x_lr.size(0), LATENT_DIM, device=x_lr.device)
            feature_cr = self.feature_decoder(z)
            kl_loss = torch.tensor(0.0, device=x_lr.device)
        
        combined_feature = feature_x + feature_cr 
        sr_image = self.image_decoder(combined_feature)
        
        return sr_image, kl_loss

# Loss Functions

In [ ]:
class MultiScalePerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True).features
        self.layer1 = vgg[:4].eval()
        self.layer2 = vgg[:9].eval()
        self.layer3 = vgg[:16].eval()
        self.layer4 = vgg[:23].eval()
        
        for param in self.parameters():
            param.requires_grad = False
        self.criterion = nn.L1Loss()
        
    def forward(self, sr, hr):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(sr.device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(sr.device)
        
        sr_norm = (sr - mean) / std
        hr_norm = (hr - mean) / std
        
        loss = 0.0
        loss += self.criterion(self.layer1(sr_norm), self.layer1(hr_norm)) * 0.15
        loss += self.criterion(self.layer2(sr_norm), self.layer2(hr_norm)) * 0.25
        loss += self.criterion(self.layer3(sr_norm), self.layer3(hr_norm)) * 0.35
        loss += self.criterion(self.layer4(sr_norm), self.layer4(hr_norm)) * 0.45
        return loss

class EdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.sobel_x = sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.sobel_y = sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.criterion = nn.L1Loss()
        
    def forward(self, sr, hr):
        if not hasattr(self, 'sobel_x_cuda'):
            self.sobel_x_cuda = self.sobel_x.to(sr.device)
            self.sobel_y_cuda = self.sobel_y.to(sr.device)
        sr_edge_x = F.conv2d(sr, self.sobel_x_cuda, padding=1, groups=3)
        sr_edge_y = F.conv2d(sr, self.sobel_y_cuda, padding=1, groups=3)
        sr_edge = torch.sqrt(sr_edge_x**2 + sr_edge_y**2 + 1e-6)
        hr_edge_x = F.conv2d(hr, self.sobel_x_cuda, padding=1, groups=3)
        hr_edge_y = F.conv2d(hr, self.sobel_y_cuda, padding=1, groups=3)
        hr_edge = torch.sqrt(hr_edge_x**2 + hr_edge_y**2 + 1e-6)
        return self.criterion(sr_edge, hr_edge)

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, size_average=True):
        super().__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 3
        self.window = self.create_window(window_size, self.channel)

    def gaussian(self, window_size, sigma):
        gauss = torch.Tensor([exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
        return gauss/gauss.sum()

    def create_window(self, window_size, channel):
        _1D_window = self.gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
        window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
        return window

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()

        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self.create_window(self.window_size, channel)
            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel

        mu1 = F.conv2d(img1, window, padding=self.window_size//2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=self.window_size//2, groups=channel)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2

        sigma1_sq = F.conv2d(img1*img1, window, padding=self.window_size//2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, window, padding=self.window_size//2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, window, padding=self.window_size//2, groups=channel) - mu1_mu2

        C1 = 0.01**2
        C2 = 0.03**2

        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))

        if self.size_average:
            return 1 - ssim_map.mean()
        else:
            return 1 - ssim_map.mean(1).mean(1).mean(1)

# MLC Model Components

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p),1).pow(1.0/self.p)

class PatchGNN(nn.Module):
    def __init__(self, in_ch, hidden=256, heads=4, out_dim=2048):
        super().__init__()
        self.enabled = USE_PYG
        if not self.enabled:
            self.fallback_conv = nn.Sequential(
                nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(in_ch),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1)
            )
            return
            
        self.lin = nn.Linear(in_ch, hidden)
        self.gat1 = GATConv(hidden, hidden // heads, heads=heads, concat=True)
        self.gat2 = GATConv(hidden, hidden // heads, heads=heads, concat=True)
        self.pool_lin = nn.Linear(hidden, out_dim)
    
    def forward(self, feat_map):
        if not self.enabled:
            patch_feat = self.fallback_conv(feat_map).flatten(1)
            return patch_feat
        
        B, C, H, W = feat_map.shape
        out_list = []
        for b in range(B):
            x = feat_map[b]
            x = x.permute(1, 2, 0).reshape(-1, C)
            x = self.lin(x)
            edge_index = grid_edge_index(H, W, x.device)
            x = F.elu(self.gat1(x, edge_index))
            x = F.elu(self.gat2(x, edge_index))
            pooled = x.mean(dim=0, keepdim=True)
            pooled = self.pool_lin(pooled)
            out_list.append(pooled)
        out = torch.cat(out_list, dim=0)
        return out

class LabelGNN(nn.Module):
    def __init__(self, num_classes, in_dim=1, hidden=128, heads=2):
        super().__init__()
        self.num_classes = num_classes
        if not USE_PYG:
            self.enabled = False
            return
        self.enabled = True
        self.fc_in = nn.Linear(in_dim, hidden)
        self.gat1 = GATConv(hidden, hidden // heads, heads=heads, concat=True)
        self.gat2 = GATConv(hidden, hidden // heads, heads=heads, concat=True)
        self.fc_out = nn.Linear(hidden, 1)
    def forward(self, logits, edge_index):
        if not self.enabled:
            return logits
        B, C = logits.shape
        out_logits = []
        for b in range(B):
            node_feat = logits[b].unsqueeze(1)
            x = F.elu(self.fc_in(node_feat))
            x = self.gat1(x, edge_index)
            x = F.elu(x)
            x = self.gat2(x, edge_index)
            refined = self.fc_out(x).squeeze(1)
            out_logits.append(refined.unsqueeze(0))
        return torch.cat(out_logits, dim=0)

class DualGNNModel(nn.Module):
    def __init__(self, num_classes, dropout=MLC_DROPOUT, use_patch_gnn=True, use_label_gnn=True):
        super().__init__()
        self.num_classes = num_classes
        
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        feat_dim = 2048
        
        self.use_patch_gnn = use_patch_gnn
        self.patch_gnn = PatchGNN(in_ch=feat_dim, hidden=256) if use_patch_gnn else None
        self.pool = GeM()
        
        if self.use_patch_gnn:
            cls_in = feat_dim * 2
        else:
            cls_in = feat_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(cls_in, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout/2),
            nn.Linear(512, num_classes)
        )
        self.label_gnn = LabelGNN(num_classes=num_classes) if use_label_gnn else None

    def forward(self, x, return_features=False):
        feat_map = self.features(x)
        pooled = self.pool(feat_map).flatten(1)
        
        if self.use_patch_gnn and self.patch_gnn is not None:
            patch_feat = self.patch_gnn(feat_map)
            combined = torch.cat([pooled, patch_feat], dim=1)
        else:
            combined = pooled
            
        logits = self.classifier(combined)
        
        if return_features:
            return logits, combined, pooled
        else:
            return logits

#  Data Loading Utilities

In [ ]:
def load_tif_as_pil(path, target_channels=3):
    try:
        if isinstance(path, str):
            path = Path(path)
            
        if not path.exists():
            return Image.new('RGB', (224, 224), (128, 128, 128))
        
        data_array = tifffile.imread(str(path))
        
        if data_array.ndim == 2:
            data_array = np.expand_dims(data_array, axis=-1)
        elif data_array.ndim == 3:
            if data_array.shape[0] < data_array.shape[1] and data_array.shape[0] < data_array.shape[2]:
                data_array = np.moveaxis(data_array, 0, -1)
        
        if data_array.shape[-1] > target_channels:
            data_array = data_array[..., :target_channels]
        elif data_array.shape[-1] < target_channels:
            missing = target_channels - data_array.shape[-1]
            duplicate_channel = data_array[..., -1:]
            data_array = np.concatenate([data_array] + [duplicate_channel] * missing, axis=-1)
        
        if data_array.dtype != np.uint8:
            max_val = data_array.max()
            if max_val > 0:
                data_array = (data_array / max_val * 255).astype(np.uint8)
            else:
                data_array = data_array.astype(np.uint8)

        return Image.fromarray(data_array).convert('RGB')
    
    except Exception as e:
        print(f"Error loading TIFF {path}: {e}")
        return Image.new('RGB', (224, 224), (128, 128, 128))

@lru_cache(maxsize=8192)
def load_image_cached(path_str: str):
    try:
        path = Path(path_str)
        if not path.exists():
            return Image.new('RGB', (224, 224), (128, 128, 128))
            
        if path.suffix.lower() in ['.tif', '.tiff']:
            return load_tif_as_pil(path)
        else:
            return Image.open(path).convert('RGB')
    except Exception as e:
        print(f"Error loading {path_str}: {e}")
        return Image.new('RGB', (224, 224), (128, 128, 128))

def find_matching_files(sentinel_path, aerial_path):
    lr_candidates = [
        SENTINEL_ROOT / sentinel_path,
        SENTINEL_ROOT / sentinel_path.upper(),
        SENTINEL_ROOT / sentinel_path.replace('img', 'IMG'),
        SENTINEL_ROOT / sentinel_path.replace('img', 'IMG_'),
    ]
    
    hr_candidates = [
        AERIAL_ROOT / aerial_path,
        AERIAL_ROOT / aerial_path.upper(),
        AERIAL_ROOT / aerial_path.replace('IMG', 'img'),
        AERIAL_ROOT / aerial_path.replace('IMG_', 'img'),
    ]
    
    lr_file, hr_file = None, None
    for candidate in lr_candidates:
        if candidate.exists():
            lr_file = candidate
            break
    
    for candidate in hr_candidates:
        if candidate.exists():
            hr_file = candidate
            break
            
    return lr_file, hr_file

# Data Loading Function

In [ ]:
def load_training_data():
    if not LABEL_PAIRS_FILE.exists():
        raise FileNotFoundError(f"Label pairs JSON not found: {LABEL_PAIRS_FILE}")
    
    with open(LABEL_PAIRS_FILE, 'r') as f:
        pairs = json.load(f)
    
    print(f"Loaded JSON with {len(pairs)} entries")
    
    all_labels = set()
    for sentinel_path, info in pairs.items():
        labels = info.get('labels', [])
        all_labels.update(map(str, labels))
    
    classes = sorted(list(all_labels))
    class2idx = {c: i for i, c in enumerate(classes)}
    print(f"Found {len(classes)} unique classes: {classes}")
    
    train_paths, train_labels, train_pairs = [], [], []
    missing_lr, missing_hr = 0, 0
    
    for sentinel_path, info in pairs.items():
        aerial_path = info.get('aerial')
        labels = info.get('labels', [])
        
        if not aerial_path:
            missing_hr += 1
            continue
            
        str_labels = [str(x) for x in labels]
        label_vec = np.zeros(len(classes), dtype=np.float32)
        for L in str_labels:
            if L in class2idx:
                label_vec[class2idx[L]] = 1.0
        
        lr_file, hr_file = find_matching_files(sentinel_path, aerial_path)
        
        if lr_file and hr_file:
            train_paths.append({
                'lr': str(lr_file), 'hr': str(hr_file),
                'sentinel_path': sentinel_path, 'aerial_path': aerial_path
            })
            train_labels.append(label_vec)
            train_pairs.append(info)
        else:
            if not lr_file: missing_lr += 1
            if not hr_file: missing_hr += 1
    
    train_labels_arr = np.array(train_labels) if train_labels else np.zeros((0, len(classes)), dtype=np.float32)
    
    print(f"\nData loading summary:")
    print(f"Successfully loaded: {len(train_paths)} pairs")
    print(f"Missing LR images: {missing_lr}")
    print(f"Missing HR images: {missing_hr}")
    
    return train_paths, train_labels_arr, classes, train_labels_arr.sum(axis=0), train_pairs

#  Dataset Classes

In [ ]:
class SRMLCDataset(Dataset):
    def __init__(self, data_paths, labels, lr_transform=None, hr_transform=None, mode='train'):
        self.data_paths = data_paths
        self.labels = labels
        self.lr_transform = lr_transform
        self.hr_transform = hr_transform
        self.mode = mode
        
    def __len__(self):
        return len(self.data_paths)
    
    def __getitem__(self, idx):
        path_info = self.data_paths[idx]
        lr_path = path_info['lr']
        hr_path = path_info['hr']
        label = self.labels[idx].copy() if len(self.labels) > 0 else np.array([])
        
        lr_img = load_image_cached(lr_path)
        hr_img = load_image_cached(hr_path)
        
        if self.lr_transform:
            lr_img_tensor = self.lr_transform(lr_img)
        else:
            lr_img_tensor = transforms.ToTensor()(lr_img)
        
        if self.hr_transform:
            hr_img_tensor = self.hr_transform(hr_img)
        else:
            hr_img_tensor = transforms.ToTensor()(hr_img)
        
        if len(label) > 0:
            label_tensor = torch.from_numpy(label).float()
        else:
            label_tensor = torch.zeros(1).float()
            
        return lr_img_tensor, hr_img_tensor, label_tensor, lr_path, hr_path

class MultiLabelDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None, use_mixup=False):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.use_mixup = use_mixup
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = load_image_cached(path)
        label = self.labels[idx].copy()
        
        if self.transform:
            img = self.transform(img)
            if self.use_mixup and random.random() < 0.2:
                k = random.randint(0, len(self.image_paths)-1)
                mix = load_image_cached(self.image_paths[k])
                mix = self.transform(mix)
                lam = random.uniform(0.3,0.7)
                img = lam * img + (1-lam) * mix
                label = lam * label + (1-lam) * self.labels[k]
        
        return img, torch.from_numpy(label).float()

class SimulateSR:
    def __init__(self, downscale_range=(1.5,3.0), jpeg_range=(70,95), blur_prob=0.4, noise_std=0.02):
        self.downscale_range = downscale_range
        self.jpeg_range = jpeg_range
        self.blur_prob = blur_prob
        self.noise_std = noise_std
        
    def __call__(self, img: Image.Image):
        w,h = img.size
        scale = random.uniform(*self.downscale_range)
        small = img.resize((max(1,int(w/scale)), max(1,int(h/scale))), resample=Image.BILINEAR)
        img = small.resize((w,h), resample=Image.BICUBIC)
        if random.random() < self.blur_prob:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3,1.2)))
        q = random.randint(*self.jpeg_range)
        from io import BytesIO
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=q)
        buf.seek(0)
        img = Image.open(buf).convert('RGB')
        if random.random() < 0.5 and self.noise_std > 0:
            arr = np.array(img).astype(np.float32) / 255.0
            arr = np.clip(arr + np.random.normal(0, self.noise_std, arr.shape), 0, 1)
            img = Image.fromarray((arr*255).astype(np.uint8))
        return img

# MLC Utilities

In [ ]:
def compute_label_cooccurrence_matrix(labels_np):
    if labels_np.size == 0:
        return np.zeros((0, 0), dtype=np.float32)

    N, C = labels_np.shape
    co = np.zeros((C, C), dtype=np.float32)
    labels_bin = (labels_np == 1).astype(np.int32)

    for i in range(C):
        denom = labels_bin[:, i].sum()
        if denom == 0:
            continue
        mask_i = labels_bin[:, i:i+1]
        both = np.sum(mask_i & labels_bin, axis=0)
        co[i, :] = both.astype(np.float32) / float(max(1, denom))

    return co

def build_label_edge_index_from_cooccur(co_mat, threshold=0.2, device='cpu'):
    C = co_mat.shape[0]
    edges = []
    for i in range(C):
        for j in range(C):
            if i == j: continue
            if co_mat[i, j] >= threshold:
                edges.append((i, j))
    if len(edges) == 0:
        edge_index = torch.zeros((2,0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index.to(device)

_grid_edge_cache = {}
def grid_edge_index(H,W,device):
    key = (H,W)
    if key in _grid_edge_cache:
        return _grid_edge_cache[key].to(device)
    edges = []
    def idx(r,c): return r*W + c
    for r in range(H):
        for c in range(W):
            u = idx(r,c)
            for dr in (-1,0,1):
                for dc in (-1,0,1):
                    if dr==0 and dc==0: continue
                    nr, nc = r+dr, c+dc
                    if 0<=nr<H and 0<=nc<W:
                        v = idx(nr,nc)
                        edges.append((u,v))
    if len(edges)==0:
        ei = torch.zeros((2,0), dtype=torch.long)
    else:
        ei = torch.tensor(edges, dtype=torch.long).t().contiguous()
    _grid_edge_cache[key] = ei
    return ei.to(device)

# Training Stages

In [ ]:
def get_kl_weight(epoch, warmup_epochs=100):
    return min(1.0, epoch / warmup_epochs) * BASE_BETA

def train_stage1_sr_only(model, train_loader, val_loader, epochs=SR_PRETRAIN_EPOCHS):
    print("=== STAGE 1: Training SR Model Only ===")
    
    pixel_loss = nn.MSELoss()
    perceptual_loss = MultiScalePerceptualLoss().to(DEVICE)
    edge_loss = EdgeLoss().to(DEVICE)
    ssim_loss = SSIMLoss().to(DEVICE)
    
    optimizer = optim.Adam(model.sr_module.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    best_psnr = 0.0
    train_losses = []
    
    for epoch in range(epochs):
        current_beta = get_kl_weight(epoch)
        
        model.train()
        model.sr_module.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f'Stage1 Epoch {epoch+1}/{epochs}')
        for lr_imgs, hr_imgs, _, _, _ in pbar:
            lr_imgs, hr_imgs = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE)
            
            optimizer.zero_grad()
            
            sr_imgs, kl_loss = model.sr_module(lr_imgs, hr_imgs)
            
            pix_l = pixel_loss(sr_imgs, hr_imgs)
            perc_l = perceptual_loss(sr_imgs, hr_imgs)
            edge_l = edge_loss(sr_imgs, hr_imgs)
            ssim_l = ssim_loss(sr_imgs, hr_imgs)
            
            if epoch < 50:
                total_loss = pix_l + 0.1 * current_beta * kl_loss + 0.05 * perc_l + 0.02 * edge_l + 0.02 * ssim_l
            else:
                total_loss = (pix_l + 
                             current_beta * kl_loss + 
                             LAMBDA_PERCEP * perc_l + 
                             LAMBDA_EDGE * edge_l + 
                             LAMBDA_SSIM * ssim_l)
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += total_loss.item()
            
            with torch.no_grad():
                batch_psnr = calculate_psnr(sr_imgs, hr_imgs).item()
            
            pbar.set_postfix({
                'Loss': f'{total_loss.item():.4f}',
                'PSNR': f'{batch_psnr:.2f}',
                'Pix': f'{pix_l.item():.4f}'
            })
        
        model.eval()
        val_psnr, val_ssim = 0.0, 0.0
        val_batches = 0
        
        with torch.no_grad():
            for lr_imgs, hr_imgs, _, _, _ in val_loader:
                lr_imgs, hr_imgs = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE)
                sr_imgs, _ = model.sr_module(lr_imgs)
                
                val_psnr += calculate_psnr(sr_imgs, hr_imgs).item()
                val_ssim += calculate_ssim(sr_imgs, hr_imgs).item()
                val_batches += 1
        
        val_psnr /= val_batches
        val_ssim /= val_batches
        avg_train_loss = train_loss / len(train_loader)
        
        train_losses.append(avg_train_loss)
        
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, '
              f'Val PSNR: {val_psnr:.2f}, Val SSIM: {val_ssim:.4f}')
        
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.sr_module.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'psnr': val_psnr,
                'ssim': val_ssim
            }, OUT_DIR / 'stage1_sr_best.pth')
            print(f"New best PSNR: {val_psnr:.2f} dB")
        
        scheduler.step()
    
    torch.save(model.sr_module.state_dict(), OUT_DIR / 'stage1_sr_final.pth')
    print(f"Stage 1 completed. Best PSNR: {best_psnr:.2f} dB")
    return train_losses

def train_stage2_mlc_only(model, train_loader, val_loader, num_classes, label_edge_index, label_gnn, epochs=MLC_TRAIN_EPOCHS):
    print("=== STAGE 2: Training MLC on HR Images ===")
    
    for param in model.sr_module.parameters():
        param.requires_grad = False
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.mlc_module.parameters(), lr=MLC_LR, weight_decay=MLC_WEIGHT_DECAY)
    
    def create_scheduler(optimizer, train_loader_len, epochs):
        total_steps = max(1, epochs * train_loader_len)
        warmup_steps = int(0.03 * total_steps)
        def lr_lambda(step):
            if step < warmup_steps:
                return float(step) / float(max(1, warmup_steps))
            progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
            return 0.5 * (1.0 + math.cos(math.pi * progress))
        return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    
    scheduler = create_scheduler(optimizer, len(train_loader), epochs)
    
    best_f1 = 0.0
    train_losses = []
    
    for epoch in range(epochs):
        model.train()
        model.sr_module.eval()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f'Stage2 Epoch {epoch+1}/{epochs}')
        all_preds, all_labels = [], []
        
        for hr_imgs, labels in pbar:
            hr_imgs, labels = hr_imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            logits = model.mlc_module(hr_imgs)
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            
            preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
            
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for hr_imgs, labels in val_loader:
                hr_imgs, labels = hr_imgs.to(DEVICE), labels.to(DEVICE)
                
                logits = model.mlc_module(hr_imgs)
                preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
                val_preds.append(preds)
                val_labels.append(labels.cpu().numpy())
        
        if all_preds and all_labels:
            train_preds = np.vstack(all_preds)
            train_labels = np.vstack(all_labels)
            train_metrics = compute_mlc_metrics(train_preds, train_labels)
        else:
            train_metrics = {'micro_f1': 0.0}
            
        if val_preds and val_labels:
            val_preds = np.vstack(val_preds)
            val_labels = np.vstack(val_labels)
            val_metrics_epoch = compute_mlc_metrics(val_preds, val_labels)
        else:
            val_metrics_epoch = {'micro_f1': 0.0}
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, '
              f'Val Micro F1: {val_metrics_epoch["micro_f1"]:.4f}')
        
        if val_metrics_epoch["micro_f1"] > best_f1:
            best_f1 = val_metrics_epoch["micro_f1"]
            torch.save({
                'epoch': epoch,
                'mlc_state_dict': model.mlc_module.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'micro_f1': best_f1,
                'metrics': val_metrics_epoch
            }, OUT_DIR / 'stage2_mlc_best.pth')
    
    torch.save(model.mlc_module.state_dict(), OUT_DIR / 'stage2_mlc_final.pth')
    print(f"Stage 2 completed. Best Micro F1: {best_f1:.4f}")
    return train_losses

# *Enhanced Joint Training* 

In [ ]:
def train_stage3_joint_improved(model, train_loader, val_loader, num_classes, epochs=JOINT_EPOCHS):
    print("=== STAGE 3: Enhanced Joint Fine-tuning ===")
    
    for param in model.sr_module.parameters():
        param.requires_grad = True
    for param in model.mlc_module.parameters():
        param.requires_grad = True
    
    sr_pixel_loss = nn.MSELoss()
    sr_perceptual_loss = MultiScalePerceptualLoss().to(DEVICE)
    sr_edge_loss = EdgeLoss().to(DEVICE)
    sr_ssim_loss = SSIMLoss().to(DEVICE)
    mlc_loss = nn.BCEWithLogitsLoss()
    feature_align_loss = nn.MSELoss()
    
    log_vars = nn.Parameter(torch.zeros(3, device=DEVICE))
    
    optimizer = optim.Adam([
        {'params': [p for n, p in model.sr_module.named_parameters() 
                   if 'vgg_encoder_r' not in n], 'lr': JOINT_LR},
        {'params': model.mlc_module.parameters(), 'lr': JOINT_LR * 0.1},
        {'params': [log_vars], 'lr': 1e-3}
    ], weight_decay=1e-5)
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=1e-7)
    
    def get_curriculum_weights(epoch, total_epochs):
        progress = epoch / total_epochs
        sr_weight = 0.7 * (1 + np.cos(np.pi * progress)) / 2 + 0.3
        mlc_weight = 1.0 - sr_weight
        return sr_weight, mlc_weight
    
    best_joint_metric = 0.0
    train_losses = []
    weight_history = {'sr': [], 'mlc': [], 'align': []}
    
    for epoch in range(epochs):
        curr_sr_weight, curr_mlc_weight = get_curriculum_weights(epoch, epochs)
        
        model.train()
        epoch_losses = {'total': 0.0, 'sr': 0.0, 'mlc': 0.0, 'align': 0.0}
        
        pbar = tqdm(train_loader, desc=f'Stage3 Epoch {epoch+1}/{epochs}')
        all_preds, all_labels = [], []
        
        for batch_idx, (lr_imgs, hr_imgs, labels, _, _) in enumerate(pbar):
            lr_imgs, hr_imgs, labels = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            sr_imgs, kl_loss = model.sr_module(lr_imgs, hr_imgs)
            
            with torch.no_grad():
                hr_features = model.sr_module.vgg_encoder_r(hr_imgs)
            sr_features = model.sr_module.vgg_encoder_x(sr_imgs)
            
            sr_logits = model.mlc_module(sr_imgs)
            with torch.no_grad():
                hr_logits = model.mlc_module(hr_imgs)
            
            kl_weight = get_kl_weight(epoch)
            pix_l = sr_pixel_loss(sr_imgs, hr_imgs)
            perc_l = sr_perceptual_loss(sr_imgs, hr_imgs)
            edge_l = sr_edge_loss(sr_imgs, hr_imgs)
            ssim_l = sr_ssim_loss(sr_imgs, hr_imgs)
            
            sr_total_loss = (pix_l + kl_weight * kl_loss + 
                           LAMBDA_PERCEP * perc_l + 
                           LAMBDA_EDGE * edge_l + 
                           LAMBDA_SSIM * ssim_l)
            
            mlc_total_loss = mlc_loss(sr_logits, labels)
            
            if sr_features.shape != hr_features.shape:
                hr_features = F.adaptive_avg_pool2d(hr_features, sr_features.shape[2:])
            align_loss = feature_align_loss(sr_features, hr_features)
            
            precision_sr = torch.exp(-log_vars[0])
            precision_mlc = torch.exp(-log_vars[1])
            precision_align = torch.exp(-log_vars[2])
            
            uncertainty_loss = (precision_sr * sr_total_loss + log_vars[0] +
                              precision_mlc * mlc_total_loss + log_vars[1] +
                              precision_align * align_loss + log_vars[2])
            
            curriculum_loss = (curr_sr_weight * sr_total_loss + 
                             curr_mlc_weight * mlc_total_loss +
                             0.1 * align_loss)
            
            blend_factor = 0.5 + 0.5 * (epoch / epochs)
            total_loss = (1 - blend_factor) * uncertainty_loss + blend_factor * curriculum_loss
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_losses['total'] += total_loss.item()
            epoch_losses['sr'] += sr_total_loss.item()
            epoch_losses['mlc'] += mlc_total_loss.item()
            epoch_losses['align'] += align_loss.item()
            
            preds = (torch.sigmoid(sr_logits) > 0.5).float().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
            
            pbar.set_postfix({
                'Total': f'{total_loss.item():.4f}',
                'SR': f'{sr_total_loss.item():.4f}',
                'MLC': f'{mlc_total_loss.item():.4f}',
                'Align': f'{align_loss.item():.4f}',
                'σ_sr': f'{torch.exp(log_vars[0]).item():.3f}',
                'σ_mlc': f'{torch.exp(log_vars[1]).item():.3f}',
                'Curr_SR': f'{curr_sr_weight:.3f}'
            })
        
        model.eval()
        val_sr_psnr, val_sr_ssim = 0.0, 0.0
        val_preds, val_labels = [], []
        val_batches = 0
        
        with torch.no_grad():
            for lr_imgs, hr_imgs, labels, _, _ in val_loader:
                lr_imgs, hr_imgs, labels = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE), labels.to(DEVICE)
                
                sr_imgs, _ = model.sr_module(lr_imgs)
                logits = model.mlc_module(sr_imgs)
                
                val_sr_psnr += calculate_psnr(sr_imgs, hr_imgs).item()
                val_sr_ssim += calculate_ssim(sr_imgs, hr_imgs).item()
                
                preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
                val_preds.append(preds)
                val_labels.append(labels.cpu().numpy())
                val_batches += 1
        
        val_sr_psnr /= val_batches
        val_sr_ssim /= val_batches
        
        if val_preds and val_labels:
            val_preds = np.vstack(val_preds)
            val_labels = np.vstack(val_labels)
            val_mlc_metrics = compute_mlc_metrics(val_preds, val_labels)
        else:
            val_mlc_metrics = {'micro_f1': 0.0}
        
        joint_metric = (0.4 * (val_sr_psnr / 50.0) + 0.6 * val_mlc_metrics['micro_f1'])
        
        for key in epoch_losses:
            epoch_losses[key] /= len(train_loader)
        
        train_losses.append(epoch_losses['total'])
        
        weight_history['sr'].append(torch.exp(-log_vars[0]).item())
        weight_history['mlc'].append(torch.exp(-log_vars[1]).item())
        weight_history['align'].append(torch.exp(-log_vars[2]).item())
        
        print(f'\nEpoch {epoch+1}/{epochs}:')
        print(f'  Losses - Total: {epoch_losses["total"]:.4f}, SR: {epoch_losses["sr"]:.4f}, '
              f'MLC: {epoch_losses["mlc"]:.4f}, Align: {epoch_losses["align"]:.4f}')
        print(f'  Uncertainty weights - σ_sr: {weight_history["sr"][-1]:.4f}, '
              f'σ_mlc: {weight_history["mlc"][-1]:.4f}')
        print(f'  Curriculum weights - SR: {curr_sr_weight:.3f}, MLC: {curr_mlc_weight:.3f}')
        print(f'  SR - PSNR: {val_sr_psnr:.2f} dB, SSIM: {val_sr_ssim:.4f}')
        print(f'  MLC - Micro F1: {val_mlc_metrics["micro_f1"]:.4f}')
        print(f'  Joint Metric: {joint_metric:.4f}')
        
        if joint_metric > best_joint_metric:
            best_joint_metric = joint_metric
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'log_vars': log_vars,
                'optimizer_state_dict': optimizer.state_dict(),
                'sr_psnr': val_sr_psnr,
                'sr_ssim': val_sr_ssim,
                'mlc_metrics': val_mlc_metrics,
                'joint_metric': joint_metric,
                'weight_history': weight_history
            }, OUT_DIR / 'stage3_joint_best.pth')
            print(f'  ✓ New best model saved (Joint Metric: {joint_metric:.4f})')
        
        scheduler.step()
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'log_vars': log_vars,
        'weight_history': weight_history
    }, OUT_DIR / 'stage3_joint_final.pth')
    
    print(f"\nStage 3 completed. Best Joint Metric: {best_joint_metric:.4f}")
    return train_losses

# Testing Functions

In [ ]:
def load_test_data_aligned_with_training(classes, class2idx):
    if not TEST_JSON_FILE.exists():
        raise FileNotFoundError(f"Test JSON file not found: {TEST_JSON_FILE}")
    
    with open(TEST_JSON_FILE, 'r') as f:
        test_data = json.load(f)
    
    print(f"Loaded test JSON with {len(test_data)} entries")
    
    test_paths, test_labels = [], []
    missing_lr, missing_hr = 0, 0
    
    for sentinel_filename, info in test_data.items():
        aerial_filename = info.get('aerial')
        labels = info.get('labels', [])
        
        if not aerial_filename:
            missing_hr += 1
            continue
            
        str_labels = [str(x) for x in labels]
        label_vec = np.zeros(len(classes), dtype=np.float32)
        for L in str_labels:
            if L in class2idx:
                label_vec[class2idx[L]] = 1.0
        
        lr_file, hr_file = None, None
        
        lr_candidate = TEST_SENTINEL_ROOT / sentinel_filename
        if lr_candidate.exists():
            lr_file = lr_candidate
        
        hr_candidate = TEST_AERIAL_ROOT / 'Z18_UN' / aerial_filename
        if hr_candidate.exists():
            hr_file = hr_candidate
        
        if lr_file and hr_file:
            test_paths.append({
                'lr': str(lr_file), 
                'hr': str(hr_file),
                'sentinel_path': sentinel_filename, 
                'aerial_path': aerial_filename
            })
            test_labels.append(label_vec)
    
    test_labels_arr = np.array(test_labels) if test_labels else np.zeros((0, len(classes)), dtype=np.float32)
    
    print(f"\nTest data loading summary:")
    print(f"Successfully loaded: {len(test_paths)} pairs")
    print(f"Missing LR images: {missing_lr}")
    print(f"Missing HR images: {missing_hr}")
    
    return test_paths, test_labels_arr

def test_final_pipeline_comprehensive(model, test_loader, num_classes, output_dir, classes):
    print("=== COMPREHENSIVE PIPELINE TESTING ON SEPARATE TEST SET ===")
    
    output_dir.mkdir(exist_ok=True)
    model.eval()
    
    sr_psnrs, sr_ssims = [], []
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch_idx, (lr_imgs, hr_imgs, labels, _, _) in enumerate(tqdm(test_loader, desc='Testing')):
            lr_imgs, hr_imgs, labels = lr_imgs.to(DEVICE), hr_imgs.to(DEVICE), labels.to(DEVICE)
            
            sr_imgs, _ = model.sr_module(lr_imgs)
            logits = model.mlc_module(sr_imgs)
            
            for j in range(len(lr_imgs)):
                sr_tensor = sr_imgs[j].unsqueeze(0)
                hr_tensor = hr_imgs[j].unsqueeze(0)
                psnr = calculate_psnr(sr_tensor, hr_tensor).item()
                ssim = calculate_ssim(sr_tensor, hr_tensor).item()
                sr_psnrs.append(psnr)
                sr_ssims.append(ssim)
            
            batch_preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()
            batch_labels = labels.cpu().numpy()
            all_preds.append(batch_preds)
            all_labels.append(batch_labels)
    
    avg_psnr = np.mean(sr_psnrs) if sr_psnrs else 0.0
    avg_ssim = np.mean(sr_ssims) if sr_ssims else 0.0
    
    if all_preds and all_labels:
        all_preds_array = np.vstack(all_preds)
        all_labels_array = np.vstack(all_labels)
        overall_mlc_metrics = compute_mlc_metrics(all_preds_array, all_labels_array)
    else:
        overall_mlc_metrics = {}
    
    print("\n" + "="*80)
    print("COMPREHENSIVE TEST RESULTS")
    print("="*80)
    
    print(f"\nSUPER RESOLUTION METRICS:")
    print(f"  Average PSNR: {avg_psnr:.2f} dB")
    print(f"  Average SSIM: {avg_ssim:.4f}")
    
    print(f"\nMULTI-LABEL CLASSIFICATION METRICS:")
    for metric, value in overall_mlc_metrics.items():
        if isinstance(value, float):
            print(f"  {metric.replace('_', ' ').title()}: {value:.4f}")
    
    results = {
        'sr_metrics': {
            'average_psnr': avg_psnr,
            'average_ssim': avg_ssim,
            'all_psnrs': sr_psnrs,
            'all_ssims': sr_ssims
        },
        'mlc_metrics': overall_mlc_metrics,
        'classes': classes
    }
    
    with open(output_dir / 'test_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    return results